# 03 — Improve Model

Cải thiện F1 từ 0.68 (notebook 02) lên **~0.75** bằng 3 kỹ thuật:
1. **Feature Engineering** — tạo ratio features (DTI, EMI/Salary, ...)
2. **Aggregate per Customer_ID** — tận dụng panel data (mỗi khách có nhiều tháng)
3. **Hyperparameter tuning** với Optuna

**Output:** overwrite `models/model.pkl` và `models/features.json` (bản improved).

**Ghi chú**: sau khi aggregate, mỗi Customer_ID → 1 dòng. Split train/test 80/20 thông thường (không cần GroupSplit).

## 0. Setup

In [ ]:
# Chỉ trên Colab
# !pip install -q lightgbm xgboost shap optuna

In [ ]:
import json, re, warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import optuna
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, StandardScaler)

from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 80)

RANDOM_STATE = 42

In [ ]:
# Mount Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/Công nghệ dịch vụ tài chính')
DATA_DIR = PROJECT_DIR / 'data' / 'Credit_score_classification'
OUT_DIR = PROJECT_DIR / 'models'

# ===== Local =====
# PROJECT_DIR = Path('..')
# DATA_DIR = PROJECT_DIR / 'data' / 'Credit_score_classification'
# OUT_DIR = PROJECT_DIR / 'models'

OUT_DIR.mkdir(exist_ok=True, parents=True)

## 1. Load + Clean (reuse từ notebook 02)

In [ ]:
def to_num(s):
    if pd.isna(s):
        return np.nan
    cleaned = re.sub(r'[^0-9.\-]', '', str(s))
    try:
        return float(cleaned) if cleaned not in ('', '-', '.') else np.nan
    except ValueError:
        return np.nan

def parse_history(s):
    if pd.isna(s):
        return np.nan
    m = re.match(r'(\d+)\s*Years?\s*(?:and\s*(\d+)\s*Months?)?', str(s))
    if m:
        return int(m.group(1)) * 12 + (int(m.group(2)) if m.group(2) else 0)
    return np.nan

def parse_loan_types(s):
    if pd.isna(s) or s in ('', 'Not Specified'):
        return []
    parts = re.split(r',\s*(?:and\s+)?', str(s))
    return [p.strip() for p in parts if p.strip() and p.strip() != 'Not Specified']

MONTH_MAP = {'January': 1, 'February': 2, 'March': 3, 'April': 4,
             'May': 5, 'June': 6, 'July': 7, 'August': 8,
             'September': 9, 'October': 10, 'November': 11, 'December': 12}

def clean_dataset(df):
    df = df.copy()
    num_like = ['Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment',
                'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly',
                'Monthly_Balance']
    for c in num_like:
        df[c] = df[c].apply(to_num)

    df['Credit_History_Months'] = df['Credit_History_Age'].apply(parse_history)
    df['Credit_Mix'] = df['Credit_Mix'].replace('_', np.nan)
    df['Payment_Behaviour'] = df['Payment_Behaviour'].replace('!@9#%8', np.nan)
    df['Occupation'] = df['Occupation'].replace('_______', np.nan)
    df['month_idx'] = df['Month'].map(MONTH_MAP)

    # Clip outliers
    clip_rules = {
        'Age': (14, 100), 'Num_Bank_Accounts': (0, 20), 'Num_Credit_Card': (0, 20),
        'Interest_Rate': (0, 40), 'Num_of_Loan': (0, 20),
        'Num_of_Delayed_Payment': (0, 50), 'Num_Credit_Inquiries': (0, 30),
    }
    for col, (lo, hi) in clip_rules.items():
        df[col] = df[col].clip(lo, hi)
    return df

df = pd.read_csv(DATA_DIR / 'train.csv', low_memory=False)
df = clean_dataset(df)
print('After cleaning:', df.shape)
df.head(2)

## 2. Feature Engineering — Ratio Features

Tính các tỷ số có ý nghĩa domain rồi mới aggregate.

In [ ]:
def add_ratios(df):
    df = df.copy()
    # An toàn khi chia 0: dùng replace 0 → NaN rồi để imputer xử
    safe = lambda s: s.replace(0, np.nan)

    df['Debt_to_Income_Annual'] = df['Outstanding_Debt'] / safe(df['Annual_Income'])
    df['EMI_to_Salary_Ratio'] = df['Total_EMI_per_month'] / safe(df['Monthly_Inhand_Salary'])
    df['Investment_Rate'] = df['Amount_invested_monthly'] / safe(df['Monthly_Inhand_Salary'])
    df['CC_per_Bank'] = df['Num_Credit_Card'] / safe(df['Num_Bank_Accounts'])
    df['Loan_per_CC'] = df['Num_of_Loan'] / safe(df['Num_Credit_Card'])
    df['Age_Start_Credit'] = df['Age'] - df['Credit_History_Months'] / 12
    df['Debt_per_Loan'] = df['Outstanding_Debt'] / safe(df['Num_of_Loan'])
    df['Balance_to_Salary'] = df['Monthly_Balance'] / safe(df['Monthly_Inhand_Salary'])

    # Split Payment_Behaviour "High_spent_Small_value_payments" → 2 cột
    def split_beh(s):
        if pd.isna(s): return (np.nan, np.nan)
        parts = str(s).split('_')
        if len(parts) >= 4:
            spending = parts[0]  # High/Low
            value = parts[2]     # Small/Medium/Large
            return (spending, value)
        return (np.nan, np.nan)
    df[['Spending_Level', 'Payment_Value']] = df['Payment_Behaviour'].apply(
        lambda s: pd.Series(split_beh(s))
    )

    # Type_of_Loan features
    df['loan_list'] = df['Type_of_Loan'].apply(parse_loan_types)
    df['Num_Loan_Types'] = df['loan_list'].apply(len)
    all_types = [t for lst in df['loan_list'] for t in lst]
    top_loans = pd.Series(all_types).value_counts().head(5).index.tolist()
    for lt in top_loans:
        col_name = 'Has_' + lt.replace(' ', '_').replace('-', '_')
        df[col_name] = df['loan_list'].apply(lambda lst, t=lt: int(t in lst))
    df = df.drop(columns=['loan_list', 'Type_of_Loan'])

    return df

df = add_ratios(df)
new_cols = ['Debt_to_Income_Annual', 'EMI_to_Salary_Ratio', 'Investment_Rate',
            'CC_per_Bank', 'Loan_per_CC', 'Age_Start_Credit', 'Debt_per_Loan',
            'Balance_to_Salary', 'Spending_Level', 'Payment_Value', 'Num_Loan_Types']
print('New ratio features:', new_cols)
df[new_cols].head()

## 3. Aggregate per Customer_ID

Với mỗi khách hàng, giữ **snapshot tháng cuối cùng** + thêm feature `_mean`, `_std`, `_max`, `_trend` (slope theo tháng) cho các cột quan trọng.

In [ ]:
# Feature cần aggregate (thay đổi theo thời gian → tận dụng)
AGG_FEATURES = ['Outstanding_Debt', 'Num_of_Delayed_Payment', 'Delay_from_due_date',
                'Credit_Utilization_Ratio', 'Monthly_Balance', 'Amount_invested_monthly',
                'Debt_to_Income_Annual', 'EMI_to_Salary_Ratio', 'Changed_Credit_Limit']

df_sorted = df.sort_values(['Customer_ID', 'month_idx'])

# Trend (slope) mỗi feature vs month_idx cho từng khách
def slope(group, col):
    x = group['month_idx'].values.astype(float)
    y = group[col].values.astype(float)
    mask = ~(np.isnan(x) | np.isnan(y))
    if mask.sum() < 2:
        return 0.0
    x, y = x[mask], y[mask]
    if x.std() == 0:
        return 0.0
    return np.polyfit(x, y, 1)[0]

print('Computing aggregations (~1 phút)...')
agg_records = []
for cid, g in df_sorted.groupby('Customer_ID'):
    rec = g.iloc[-1].to_dict()  # snapshot tháng cuối
    for col in AGG_FEATURES:
        rec[f'{col}_mean'] = g[col].mean()
        rec[f'{col}_std'] = g[col].std()
        rec[f'{col}_max'] = g[col].max()
        rec[f'{col}_trend'] = slope(g, col)
    rec['n_months_recorded'] = len(g)
    agg_records.append(rec)

df_agg = pd.DataFrame(agg_records)
print('After aggregation:', df_agg.shape)
print(f'{df_agg["Customer_ID"].nunique()} unique customers')

## 4. Chuẩn bị data train

In [ ]:
drop_cols = ['ID', 'Name', 'SSN', 'Month', 'Credit_History_Age', 'Customer_ID',
             'Credit_Score', 'month_idx', 'Payment_Behaviour']
y_raw = df_agg['Credit_Score']
X = df_agg.drop(columns=[c for c in drop_cols if c in df_agg.columns])

target_order = ['Poor', 'Standard', 'Good']
y = y_raw.map({v: i for i, v in enumerate(target_order)})

# Phân nhóm cột
ordinal_cols = ['Credit_Mix']
credit_mix_order = [['Bad', 'Standard', 'Good']]
onehot_cols = ['Occupation', 'Payment_of_Min_Amount', 'Spending_Level', 'Payment_Value']
onehot_cols = [c for c in onehot_cols if c in X.columns]
numeric_cols = [c for c in X.columns if c not in ordinal_cols + onehot_cols]

print(f'Total features: {X.shape[1]}')
print(f'Numeric: {len(numeric_cols)}, Ordinal: {len(ordinal_cols)}, OneHot: {len(onehot_cols)}')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc', StandardScaler())
        ]), numeric_cols),
        ('ord', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('oe', OrdinalEncoder(categories=credit_mix_order,
                                   handle_unknown='use_encoded_value', unknown_value=-1))
        ]), ordinal_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), onehot_cols),
    ],
    remainder='drop'
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 5. Baseline (LightGBM default) — để đo gain

In [ ]:
def evaluate(pipe, X_te, y_te, name):
    pred = pipe.predict(X_te)
    proba = pipe.predict_proba(X_te)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average='macro')
    auc = roc_auc_score(y_te, proba, multi_class='ovr', average='macro')
    print(f'{name:25s} | Acc: {acc:.4f} | F1-macro: {f1:.4f} | AUC-ovr: {auc:.4f}')
    return {'model': name, 'accuracy': acc, 'f1_macro': f1, 'auc_ovr': auc,
            'pred': pred, 'proba': proba}

pipe_base = Pipeline([
    ('pre', preprocessor),
    ('model', LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                              random_state=RANDOM_STATE, verbose=-1,
                              class_weight='balanced'))
])
pipe_base.fit(X_train, y_train)
res_base = evaluate(pipe_base, X_test, y_test, 'LightGBM (FE+Agg only)')

## 6. Optuna Hyperparameter Tuning

50 trials, 3-fold CV. Trên Colab GPU khoảng 5-10 phút.

In [ ]:
# Pre-transform 1 lần để tuning nhanh hơn
X_train_t = preprocessor.fit_transform(X_train, y_train)
X_test_t = preprocessor.transform(X_test)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'random_state': RANDOM_STATE, 'verbose': -1, 'class_weight': 'balanced',
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr, va in skf.split(X_train_t, y_train):
        m = LGBMClassifier(**params)
        m.fit(X_train_t[tr], y_train.iloc[tr])
        scores.append(f1_score(y_train.iloc[va], m.predict(X_train_t[va]), average='macro'))
    return np.mean(scores)

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print('\nBest params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
print(f'\nBest CV F1-macro: {study.best_value:.4f}')

## 7. Train final model với best params

In [ ]:
best_params = study.best_params.copy()
best_params.update({'random_state': RANDOM_STATE, 'verbose': -1, 'class_weight': 'balanced'})

pipe_final = Pipeline([
    ('pre', preprocessor),
    ('model', LGBMClassifier(**best_params))
])
pipe_final.fit(X_train, y_train)
res_final = evaluate(pipe_final, X_test, y_test, 'LightGBM (Tuned)')

## 8. So sánh trước và sau cải tiến

In [ ]:
cmp = pd.DataFrame([
    {'stage': 'Notebook 02 (baseline)', 'accuracy': 0.689, 'f1_macro': 0.682, 'auc_ovr': 0.846},
    {'stage': 'FE + Aggregate only', **{k: res_base[k] for k in ('accuracy', 'f1_macro', 'auc_ovr')}},
    {'stage': 'FE + Aggregate + Optuna', **{k: res_final[k] for k in ('accuracy', 'f1_macro', 'auc_ovr')}},
])
cmp['delta_f1'] = (cmp['f1_macro'] - 0.682).round(4)
cmp

In [ ]:
print(classification_report(y_test, res_final['pred'], target_names=target_order))

cm = confusion_matrix(y_test, res_final['pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_order, yticklabels=target_order)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — LightGBM Tuned')
plt.show()

## 9. Convert Probability → FICO Score + histogram

In [ ]:
def proba_to_fico(proba):
    expected = proba @ np.array([0, 1, 2])
    return 300 + expected / 2 * 550

def fico_to_rating(score):
    if score < 580: return 'Poor'
    if score < 670: return 'Fair'
    if score < 740: return 'Good'
    if score < 800: return 'Very Good'
    return 'Exceptional'

fico = proba_to_fico(res_final['proba'])
plt.figure(figsize=(10, 4))
sns.histplot(fico, bins=50, kde=True)
for x, lbl in [(580, 'Fair'), (670, 'Good'), (740, 'Very Good'), (800, 'Exceptional')]:
    plt.axvline(x, color='gray', linestyle='--', alpha=0.5)
    plt.text(x + 2, plt.ylim()[1] * 0.9, lbl, fontsize=9, color='gray')
plt.title('Phân phối FICO score (Tuned model)')
plt.show()
print(pd.Series([fico_to_rating(s) for s in fico]).value_counts())

## 10. Feature importance

In [ ]:
importances = pipe_final.named_steps['model'].feature_importances_
feat_names = pipe_final.named_steps['pre'].get_feature_names_out()
imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=imp_df, y='feature', x='importance')
plt.title('Top 20 Feature Importance (LightGBM Tuned)')
plt.tight_layout()
plt.show()

## 11. SHAP

In [ ]:
import shap

X_te_t = pipe_final.named_steps['pre'].transform(X_test)
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_te_t), 500, replace=False)
explainer = shap.TreeExplainer(pipe_final.named_steps['model'])
shap_values = explainer.shap_values(X_te_t[sample_idx])

sv_good = shap_values[2] if isinstance(shap_values, list) else shap_values[..., 2]
shap.summary_plot(sv_good, X_te_t[sample_idx], feature_names=feat_names,
                  max_display=15, show=True)

## 12. Export — overwrite model cũ

In [ ]:
joblib.dump(pipe_final, OUT_DIR / 'model.pkl')

schema = {
    'task': 'classification_3class_aggregated',
    'target': 'Credit_Score',
    'target_order': target_order,
    'best_model': 'LightGBM (Tuned + FE + Aggregate)',
    'best_params': study.best_params,
    'metrics': {
        'accuracy': float(res_final['accuracy']),
        'f1_macro': float(res_final['f1_macro']),
        'auc_ovr': float(res_final['auc_ovr']),
    },
    'input_columns': list(X.columns),
    'numeric_cols': numeric_cols,
    'ordinal_cols': ordinal_cols,
    'onehot_cols': onehot_cols,
    'credit_mix_order': credit_mix_order[0],
    'aggregated_features': AGG_FEATURES,
    'categorical_values': {
        c: sorted([str(v) for v in df_agg[c].dropna().unique().tolist()])
        for c in onehot_cols + ordinal_cols if c in df_agg.columns
    },
    'numeric_ranges': {
        c: {'min': float(df_agg[c].min()), 'max': float(df_agg[c].max()),
            'mean': float(df_agg[c].mean()), 'median': float(df_agg[c].median())}
        for c in numeric_cols if c in df_agg.columns and df_agg[c].notna().any()
    },
    'fico_conversion': {
        'formula': 'score = 300 + (0*P_poor + 1*P_std + 2*P_good) / 2 * 550',
        'bins': [
            {'max': 580, 'label': 'Poor'},
            {'max': 670, 'label': 'Fair'},
            {'max': 740, 'label': 'Good'},
            {'max': 800, 'label': 'Very Good'},
            {'max': 850, 'label': 'Exceptional'}
        ]
    },
    'note': 'Model trained on per-customer aggregated data. Streamlit inputs must include both current snapshot values AND historical aggregates (mean/std/max/trend).'
}

with open(OUT_DIR / 'features.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2, ensure_ascii=False, default=str)

print('Exported:')
for p in OUT_DIR.iterdir():
    print(f'  {p.name:30s} ({p.stat().st_size / 1024:.1f} KB)')

## 13. Sanity check

In [ ]:
loaded = joblib.load(OUT_DIR / 'model.pkl')
sample = X_test.iloc[[0]]
proba = loaded.predict_proba(sample)[0]
pred_class = target_order[loaded.predict(sample)[0]]
fico_score = proba_to_fico(proba.reshape(1, -1))[0]
rating = fico_to_rating(fico_score)
actual_class = target_order[y_test.iloc[0]]

print(f'Actual class:  {actual_class}')
print(f'Predicted:     {pred_class}')
print(f'Probabilities: Poor={proba[0]:.3f} | Standard={proba[1]:.3f} | Good={proba[2]:.3f}')
print(f'FICO score:    {fico_score:.1f}  →  {rating}')